# LLM Serving Frameworks: Which One, For What Purpose

A theory-only reference mapping each major LLM serving framework to the situation it
was actually built for — hardware target, traffic pattern, and workload shape. Builds
on the core four (vLLM, llama.cpp, Ollama, TGI) and extends into the wider 2026
serving landscape: SGLang, TensorRT-LLM, LMDeploy, MLC LLM, DeepSpeed-Inference, MLX,
NVIDIA NIM, and the orchestration layer that sits above all of them.

**Contents**
1. The four decision factors
2. Framework selection matrix (core four, extended)
3. Detailed breakdown: the core four
4. Extra frameworks worth knowing
5. The orchestration layer (above the serving engine)
6. 2026 scenario-to-framework map
7. Key trend: TGI's maintenance mode


## 1. The Four Decision Factors

Before picking a framework, it helps to evaluate it against the same four axes every
time, rather than chasing whichever benchmark headline is newest:

1. **Throughput under concurrency** — tokens/second once you have 50–100+ simultaneous
   users, not single-request latency.
2. **Hardware flexibility** — NVIDIA-only vs. multi-platform (AMD, Apple Silicon, CPU).
3. **Workload fit** — batch processing vs. multi-turn chat vs. structured output
   (JSON/code generation) vs. agentic tool-calling loops.
4. **Operational cost** — setup/compilation time, vendor lock-in, and how much DevOps
   effort the framework demands before it's production-ready.


## 2. Framework Selection Matrix

| Serving Framework | Primary Use Case | Best Hardware | Key Strength |
|---|---|---|---|
| **vLLM** | High-throughput enterprise production serving | Data center GPUs (NVIDIA A100/H100, AMD) | PagedAttention — maximum requests per second |
| **llama.cpp** | Local, resource-constrained, or CPU-only execution | Consumer hardware (laptops, CPUs, Apple Silicon) | Pure C/C++, ultra-low memory footprint |
| **Ollama** | Local development, prototyping, internal tooling | Local desktops/laptops (Mac, Windows, Linux) | Zero-config, package-manager style (`ollama run`) |
| **TGI** (Hugging Face) | Enterprise deployment — now in maintenance mode | Data center GPUs | Native HF integration, production-tested (migrate away for new projects) |
| **SGLang** | Multi-turn chat, RAG, agentic workloads with shared context | Data center GPUs (NVIDIA + AMD) | RadixAttention — reuses shared-prefix computation |
| **TensorRT-LLM** | Absolute lowest latency on NVIDIA-only infrastructure | NVIDIA GPUs exclusively | Hand-compiled, hardware-level optimization |
| **LMDeploy** | Compression + serving, TurboMind engine | NVIDIA GPUs | Strong quantization/compression focus, lower-barrier Python path |
| **MLC LLM** | Edge, mobile, and browser (WebGPU) deployment | Phones, browsers, edge devices | Truly cross-platform compiled inference |
| **DeepSpeed-Inference / MII** | Large distributed inference across many GPUs | Multi-GPU / multi-node clusters | Model/tensor parallelism at very large scale |
| **MLX (oMLX)** | Local inference on Apple Silicon | Mac (M-series) | Built for unified memory architecture |


## 3. Detailed Breakdown: The Core Four

### vLLM — Maximizing throughput in production
- **Purpose**: squeeze the maximum Requests Per Second (RPS) out of expensive cloud
  GPUs when serving many concurrent users.
- **Why it works**: LLMs waste a lot of GPU memory on KV (Key-Value) caches. vLLM's
  **PagedAttention** manages that memory the way an operating system manages virtual
  memory pages, cutting fragmentation to nearly zero.
- **Best scenario**: a commercial SaaS product or customer-facing chatbot with
  hundreds of simultaneous users where GPU cost efficiency matters.

### llama.cpp — Running anywhere without a data-center GPU
- **Purpose**: efficient LLM inference on consumer hardware, edge devices, or CPU-only
  systems.
- **Why it works**: pure C/C++ with no heavy dependencies, built around the
  **GGUF** quantized format, and capable of dynamically offloading model layers
  between CPU and system RAM. Especially well optimized for Apple Silicon via Metal.
- **Best scenario**: running a large (e.g. 70B parameter) model locally on a MacBook
  Pro, deploying to an industrial edge gateway, or serving from plain CPU servers.

### Ollama — The "it just works" local layer
- **Purpose**: simplifying local developer setup and rapid prototyping.
- **Why it works**: wraps llama.cpp in a friendly background service — acts like
  Docker for LLMs, handling model downloads, configuration, and exposing a clean
  OpenAI-compatible API with a single command.
- **Best scenario**: a software engineer building a local AI agent or internal tool
  who wants something that works out of the box, without wrestling GPU drivers.

### TGI (Text Generation Inference) — Enterprise, HF-native (now legacy-leaning)
- **Purpose**: secure, reliable model deployment inside regulated enterprise clouds.
- **Why it works**: purpose-built by Hugging Face with tensor parallelism (splitting
  models across multiple GPUs), token streaming via Server-Sent Events, and
  production-grade telemetry.
- **Best scenario historically**: an enterprise DevOps team deploying inside a
  regulated Kubernetes cluster (e.g. Amazon EKS) that needed hardened, native Hugging
  Face Hub integration. **Note**: see Section 7 — this is no longer the recommended
  starting point for new deployments.


## 4. Extra Frameworks Worth Knowing

Beyond the core four, several other engines have carved out specific niches in the
2026 serving landscape.

### SGLang — Built for shared-context workloads
- **Purpose**: high-performance serving for workloads where requests share context —
  multi-turn chat, RAG pipelines, and agent loops.
- **Why it works**: its core trick, **RadixAttention**, caches shared computation
  across requests via prefix caching, so identical context isn't recomputed. Also
  supports continuous batching, speculative decoding, tensor parallelism, chunked
  prefill, and structured-output generation (JSON/code) with quantization support
  (FP8/INT4/AWQ/GPTQ).
- **Best scenario**: chatbots, RAG systems, or agentic applications where a large
  fraction of each request's context overlaps with previous requests — this is where
  it measurably outperforms vLLM on throughput.

### TensorRT-LLM (NVIDIA) — Maximum performance, single-vendor lock-in
- **Purpose**: the absolute lowest time-to-first-token and highest throughput
  achievable on NVIDIA hardware.
- **Why it works**: models are compiled ahead-of-time into hardware-specific engines,
  squeezing out overhead that generic runtimes carry.
- **Trade-off**: demands significant setup time (often 1–2 weeks of compilation and
  tuning work) and locks you into NVIDIA exclusively. **NVIDIA NIM** offers a turnkey
  alternative — a pre-built container bundling the engine, weights, and API — for
  teams that want TensorRT-LLM's performance without the manual compilation step.

### LMDeploy — Compression-first serving
- **Purpose**: a toolkit focused on compressing, deploying, and serving LLMs, built
  around the **TurboMind** engine.
- **Best scenario**: teams that want strong quantization/compression alongside
  serving, or prefer a lower-barrier Python-first engine path as a vLLM alternative.

### MLC LLM — True cross-platform compiled inference
- **Purpose**: deploying models to edge devices, mobile phones, and browsers.
- **Why it works**: compiles models to run via WebGPU and other portable backends,
  reaching environments none of the GPU-cluster-focused frameworks can touch.
- **Best scenario**: an LLM feature that needs to run inside a mobile app or directly
  in a user's browser tab, with no server round-trip.

### DeepSpeed-Inference / DeepSpeed-MII — Large-scale distributed inference
- **Purpose**: serving very large models split across many GPUs or multiple nodes.
- **Best scenario**: models too large to fit a single GPU's memory even after
  quantization, requiring model or tensor parallelism across a cluster.

### MLX / oMLX — The Mac-native option
- **Purpose**: local inference optimized specifically for Apple Silicon's unified
  memory architecture.
- **Best scenario**: developers standardized on Mac hardware who want performance
  closer to native than llama.cpp's more general CPU/GPU offload path provides.

### Smaller/niche mentions
- **Mistral.rs** — a Rust-based local inference engine, another option in the
  local-desktop category alongside llama.cpp and Ollama.
- **OpenLLM** — serving built around the BentoML deployment ecosystem.
- **LocalAI / PowerInfer / Inferflow** — narrower tools that fit specific hardware or
  simplicity constraints; worth evaluating only when their particular assumptions
  match your project.


## 5. The Orchestration Layer (Above the Serving Engine)

Serving engines (vLLM, SGLang, TGI, LMDeploy) and hardware-optimized engines
(TensorRT-LLM, llama.cpp) sit at the model-execution level. Above them sits an
**orchestration layer** that routes traffic across multiple serving instances,
handles failover, and manages autoscaling:

- **Ray Serve** — general-purpose model-serving orchestration built on Ray,
  commonly used to scale out multiple vLLM/SGLang replicas.
- **NVIDIA Triton Inference Server** — a model-serving orchestrator that can host
  multiple frameworks/backends behind one unified serving layer.
- **NVIDIA Dynamo** — newer NVIDIA orchestration tooling aimed at large-scale,
  multi-instance LLM deployments.

These matter once you've outgrown a single serving instance and need routing,
load balancing, or multi-model hosting — they don't replace vLLM/SGLang/etc., they
sit in front of them.


## 6. 2026 Scenario-to-Framework Map

A condensed decision map reflecting where the ecosystem has settled as of 2026:

| Scenario | Recommended framework |
|---|---|
| Cloud, high-concurrency, general-purpose | **vLLM** |
| Cloud, absolute lowest latency, NVIDIA-only | **TensorRT-LLM** (or NVIDIA NIM for turnkey) |
| Multi-turn chat / RAG / agent workloads with shared context | **SGLang** |
| Mac-based local development or serving | **MLX (oMLX)**, or llama.cpp/Ollama |
| Quick local testing / prototyping, any OS | **Ollama** |
| Mobile / browser / edge deployment | **MLC LLM** |
| CPU-only or resource-constrained hardware | **llama.cpp** |
| Very large models needing multi-GPU/multi-node parallelism | **DeepSpeed-Inference/MII** |
| Compression-focused serving, alternative to vLLM | **LMDeploy** |

**Practical note**: benchmark claims vary by model, tokenizer, prompt length,
concurrency level, quantization format, and target hardware — the general mapping
above is a starting point, not a substitute for testing against your actual workload.


## 7. Key Trend: TGI's Maintenance Mode

Hugging Face's own documentation now describes **TGI as being in maintenance mode**,
and it was archived in March 2026. Hugging Face itself now recommends **vLLM or
SGLang** for new production deployments instead.

**Practical implication**: if you're maintaining an existing TGI deployment, this is
effectively a migration signal — new projects should default to vLLM (general
high-throughput production) or SGLang (shared-context / agentic workloads) rather than
building fresh on TGI.
